# 05 — Business Analysis & Insights

This notebook queries the Gold layer to produce insights a real SA bank's analytics team would care about.

We answer:
1. Which customer segment spends the most?
2. What does spending look like across SA provinces?
3. Which merchants dominate SA retail banking transactions?
4. How many customers show signs of financial stress?
5. What does the month-end spending squeeze look like?

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker

GOLD_PATH = 'dbfs:/FileStore/sa-banking-pipeline/gold'

monthly = spark.read.format('delta').load(f'{GOLD_PATH}/customer_monthly_summary')
profile = spark.read.format('delta').load(f'{GOLD_PATH}/customer_profile')

print(f'Monthly summary rows: {monthly.count():,}')
print(f'Customer profiles: {profile.count():,}')

## Analysis 1 — Average monthly spend by customer segment

In [ ]:
seg_spend = profile.groupBy('segment').agg(
    F.round(F.avg('lifetime_spend') / 24, 2).alias('avg_monthly_spend'),
    F.count('customer_id').alias('customer_count')
).toPandas().sort_values('avg_monthly_spend', ascending=False)

fig, ax = plt.subplots(figsize=(10, 5))
bars = ax.bar(seg_spend['segment'], seg_spend['avg_monthly_spend'], color=['#1a73e8','#34a853','#fbbc05','#ea4335'])
ax.set_title('Average Monthly Spend by SA Customer Segment (ZAR)', fontsize=14, fontweight='bold')
ax.set_ylabel('Average Monthly Spend (ZAR)')
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'R{x:,.0f}'))
for bar, val in zip(bars, seg_spend['avg_monthly_spend']):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 50, f'R{val:,.0f}', ha='center', fontsize=10)
plt.tight_layout()
plt.show()
print(seg_spend.to_string(index=False))

## Analysis 2 — Financial stress by province

Financial stress = more than 60% of monthly spending happens after the 20th of the month (running out of money before payday).

In [ ]:
stress_province = profile.groupBy('province').agg(
    F.round(F.avg('avg_stress_ratio') * 100, 1).alias('avg_stress_pct'),
    F.sum(F.col('is_financially_stressed').cast('int')).alias('stressed_customers'),
    F.count('customer_id').alias('total_customers')
).toPandas().sort_values('avg_stress_pct', ascending=True)

fig, ax = plt.subplots(figsize=(10, 6))
ax.barh(stress_province['province'], stress_province['avg_stress_pct'], color='#ea4335')
ax.set_title('Average Month-End Financial Stress by SA Province (%)', fontsize=13, fontweight='bold')
ax.set_xlabel('Avg % of Spend Occurring After 20th of Month')
ax.axvline(x=60, color='black', linestyle='--', label='Stress threshold (60%)')
ax.legend()
plt.tight_layout()
plt.show()
print(stress_province.to_string(index=False))

## Analysis 3 — Top spending categories across all customers

In [ ]:
from pyspark.sql import functions as F

SILVER_PATH = 'dbfs:/FileStore/sa-banking-pipeline/silver'
txn = spark.read.format('delta').load(f'{SILVER_PATH}/transactions')

cat_totals = txn.filter(F.col('transaction_type') == 'DEBIT') \
    .groupBy('merchant_category') \
    .agg(F.round(F.sum('amount') / 1e6, 2).alias('total_spend_millions')) \
    .orderBy(F.desc('total_spend_millions')).toPandas()

fig, ax = plt.subplots(figsize=(12, 5))
ax.bar(cat_totals['merchant_category'], cat_totals['total_spend_millions'], color='#1a73e8')
ax.set_title('Total Spend by Category — All SA Customers (R Millions)', fontsize=13, fontweight='bold')
ax.set_ylabel('Total Spend (R Millions)')
ax.set_xlabel('Merchant Category')
plt.xticks(rotation=30, ha='right')
plt.tight_layout()
plt.show()

## Analysis 4 — Monthly transaction volume trend (2023 vs 2024)

In [ ]:
monthly_vol = txn.groupBy('txn_year', 'txn_month').agg(
    F.count('transaction_id').alias('txn_count'),
    F.round(F.sum('amount') / 1e6, 2).alias('total_volume_millions')
).orderBy('txn_year', 'txn_month').toPandas()

monthly_vol['period'] = monthly_vol['txn_year'].astype(str) + '-' + monthly_vol['txn_month'].astype(str).str.zfill(2)

fig, ax1 = plt.subplots(figsize=(14, 5))
ax2 = ax1.twinx()
ax1.bar(monthly_vol['period'], monthly_vol['txn_count'], color='#1a73e8', alpha=0.6, label='Transaction Count')
ax2.plot(monthly_vol['period'], monthly_vol['total_volume_millions'], color='#ea4335', linewidth=2, marker='o', label='Volume (R Millions)')
ax1.set_title('Monthly Transaction Count and Volume — SA Banking Pipeline', fontsize=13, fontweight='bold')
ax1.set_xlabel('Month')
ax1.set_ylabel('Transaction Count', color='#1a73e8')
ax2.set_ylabel('Volume (R Millions)', color='#ea4335')
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.show()

## Analysis 5 — Financial stress summary

In [ ]:
total = profile.count()
stressed = profile.filter(F.col('is_financially_stressed') == True).count()
pct = round(stressed / total * 100, 1)

print('=== SA Customer Financial Stress Report ===')
print(f'Total customers analysed:  {total:,}')
print(f'Financially stressed:       {stressed:,} ({pct}%)')
print(f'Not stressed:               {total - stressed:,} ({100 - pct}%)')
print()
print('Stress by segment:')
profile.groupBy('segment').agg(
    F.sum(F.col('is_financially_stressed').cast('int')).alias('stressed'),
    F.count('customer_id').alias('total'),
    F.round(F.avg('avg_stress_ratio') * 100, 1).alias('avg_stress_pct')
).show()

## ✅ Pipeline Complete

This project demonstrates a full production-style data engineering pipeline:

| Stage | What Was Built |
|---|---|
| Data Generation | 500k synthetic SA banking transactions with realistic patterns |
| Bronze | Raw data landed into Delta Lake with audit trail |
| Silver | Cleaned, typed, deduplicated with SA-specific feature flags |
| Gold | Customer monthly summaries and financial stress profiles |
| Analysis | Business insights across segments, provinces, and categories |

**Stack:** Databricks · PySpark · Delta Lake · Python · GitHub Actions

**Author:** Morobang Tshigidimisa — [github.com/Morobang](https://github.com/Morobang)